# Notebook 5: Cell Morphology Analysis

This notebook quantifies cell morphology from Xenium cell boundary polygons, computing shape metrics such as roundness (circularity), aspect ratio, and cell area for each cell type.

**Overview:**
1. Load cell boundary polygons from Xenium output
2. Compute roundness (circularity): 4π × area / perimeter²
3. Compute aspect ratio via PCA on cell boundary coordinates
4. Compute cell area
5. Morphology comparison across cell types, subclusters, and response groups
6. Longitudinal morphology changes

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
from shapely.geometry import Polygon
from sklearn.decomposition import PCA
import scanpy as sc
import anndata
import glob
import os

plt.rcParams['figure.dpi'] = 150

## 2. Patient Metadata

In [ ]:
sample_name_ls = [
    'Pt-1_Pre',  'Pt-1_Resection',
    'Pt-2_Pre',  'Pt-2_Resection',
    'Pt-3_Pre',  'Pt-3_JustAfter',  'Pt-3_Resection',
    'Pt-4_Pre',  'Pt-4_Resection',
    'Pt-5_Pre',  'Pt-5_JustAfter',  'Pt-5_Resection',
    'Pt-6_Pre',  'Pt-6_Resection',
    'Pt-7_Pre',  'Pt-7_Resection',
    'Pt-8_Pre',  'Pt-8_Resection',
    'Pt-9_Pre',  'Pt-9_Resection',
    'Pt-10_Pre', 'Pt-10_Resection',
    'Pt-11_Pre', 'Pt-11_JustAfter', 'Pt-11_Resection',
    'Pt-12_Pre', 'Pt-12_Resection',
    'Pt-13_Pre', 'Pt-13_JustAfter', 'Pt-13_Resection',
    'Pt-14_Pre', 'Pt-14_Resection',
    'Pt-15_Pre', 'Pt-15_Resection',
    'Pt-16_Pre', 'Pt-16_Resection',
    'Pt-17_Pre', 'Pt-17_Resection',
    'Pt-18_Pre', 'Pt-18_Resection',
    'Pt-19_Pre', 'Pt-19_Resection',
    'Pt-20_Pre', 'Pt-20_Resection',
    'Pt-21_Pre', 'Pt-21_Resection',
    'Pt-22_Pre', 'Pt-22_Resection',
    'Pt-23_Pre', 'Pt-23_Resection',
    'Pt-24_Pre', 'Pt-24_Resection',
]

def get_response(pt_num):
    if pt_num <= 8:
        return 'SD'
    elif pt_num <= 16:
        return 'MPR'
    else:
        return 'CR'

## 3. Morphology Feature Computation

Three metrics are computed from the cell boundary polygon coordinates:

- **Roundness (circularity)**: `4π × area / perimeter²` — equals 1 for a perfect circle, approaches 0 for highly elongated cells
- **Aspect ratio**: ratio of major to minor axis lengths, computed via PCA on the polygon vertex coordinates
- **Cell area**: total polygon area in μm²

In [ ]:
def compute_cell_morphology(boundary_df):
    """Compute morphology features for all cells from boundary coordinate table.
    
    Parameters
    ----------
    boundary_df : DataFrame with columns [cell_id, vertex_x, vertex_y]
    
    Returns
    -------
    DataFrame with columns [cell_id, roundness, aspect_ratio, area]
    """
    records = []
    
    for cell_id, grp in boundary_df.groupby('cell_id'):
        coords = grp[['vertex_x', 'vertex_y']].values
        
        if len(coords) < 3:
            continue
        
        # Create polygon for area and perimeter
        poly = Polygon(coords)
        if not poly.is_valid:
            poly = poly.buffer(0)  # Fix invalid geometry
        
        area = poly.area
        perimeter = poly.length
        
        # Roundness (circularity): 1 = perfect circle
        if perimeter > 0:
            roundness = (4 * np.pi * area) / (perimeter ** 2)
        else:
            roundness = np.nan
        
        # Aspect ratio via PCA on boundary vertices
        if len(coords) >= 2:
            pca = PCA(n_components=2)
            pca.fit(coords)
            variances = pca.explained_variance_
            if variances[1] > 0:
                aspect_ratio = np.sqrt(variances[0]) / np.sqrt(variances[1])
            else:
                aspect_ratio = np.nan
        else:
            aspect_ratio = np.nan
        
        records.append({
            'cell_id': cell_id,
            'roundness': roundness,
            'aspect_ratio': aspect_ratio,
            'area': area,
        })
    
    return pd.DataFrame(records).set_index('cell_id')

## 4. Load Cell Boundaries and Compute Morphology

In [ ]:
# Base directory for Xenium output (adjust path as needed)
XENIUM_BASE_DIR = '../data/xenium_output'

morphology_records = []

for sample_name in sample_name_ls:
    boundary_path = os.path.join(XENIUM_BASE_DIR, sample_name, 'cell_boundaries.csv.gz')
    cells_path = os.path.join(XENIUM_BASE_DIR, sample_name, 'cells.csv.gz')
    
    if not os.path.exists(boundary_path):
        print(f'Boundary file not found: {sample_name}')
        continue
    
    # Load cell boundary coordinates
    df_boundaries = pd.read_csv(boundary_path, compression='gzip')
    
    # Compute morphology features
    df_morph = compute_cell_morphology(df_boundaries)
    df_morph['Sample'] = sample_name
    pt_num = int(sample_name.split('_')[0].replace('Pt-', ''))
    df_morph['Response'] = get_response(pt_num)
    df_morph['Timepoint'] = sample_name.split('_', 1)[1]
    df_morph['Patient'] = f'Pt-{pt_num}'
    
    morphology_records.append(df_morph)
    print(f'Processed {sample_name}: {len(df_morph)} cells')

if morphology_records:
    df_morph_all = pd.concat(morphology_records)
    print(f'\nTotal cells with morphology: {len(df_morph_all)}')

### 4.1 Load Nucleus and Cell Area from cells.csv.gz

Xenium `cells.csv.gz` contains `nucleus_area` and `cell_area` columns computed by the Xenium instrument.

In [ ]:
# Load nucleus_area and cell_area from Xenium cells.csv.gz
area_records = []

for sample_name in sample_name_ls:
    cells_path = os.path.join(XENIUM_BASE_DIR, sample_name, 'cells.csv.gz')
    if not os.path.exists(cells_path):
        print(f'cells.csv.gz not found: {sample_name}')
        continue

    df_cells = pd.read_csv(cells_path, compression='gzip')
    df_cells = df_cells[['cell_id', 'nucleus_area', 'cell_area']].copy()
    df_cells['cell_id'] = df_cells['cell_id'].astype(str) + '_' + sample_name
    df_cells['Sample'] = sample_name
    pt_num = int(sample_name.split('_')[0].replace('Pt-', ''))
    df_cells['Response'] = get_response(pt_num)
    df_cells['Timepoint'] = sample_name.split('_', 1)[1]
    area_records.append(df_cells)
    print(f'Loaded area data: {sample_name} ({len(df_cells)} cells)')

if area_records:
    df_area_all = pd.concat(area_records, ignore_index=True).set_index('cell_id')
    df_area_all['nucleus_area_log10'] = np.log10(df_area_all['nucleus_area'].clip(lower=1e-6))
    df_area_all['cell_area_log10'] = np.log10(df_area_all['cell_area'].clip(lower=1e-6))
    df_area_all.to_csv('../results/cell_nucleus_area_all.csv')
    print(f'Saved cell_nucleus_area_all.csv: {df_area_all.shape}')

In [ ]:
# Load cell-level annotations (subcluster labels) for each sample
adata = sc.read_h5ad('../data/bicrc_integrated_annotated_.h5ad')

# Create cell ID to subcluster mapping
# Cell IDs in the boundary files match the cell_id column in cells.csv.gz
cell_annotation = adata.obs[['Sample', 'subcluster', 'Major_cluster_pathol']].copy()
cell_annotation.index.name = 'cell_id'

# Merge morphology with cell type annotations
if 'df_morph_all' in dir():
    df_morph_annotated = df_morph_all.join(cell_annotation[['subcluster', 'Major_cluster_pathol']], how='left')
    df_morph_annotated.to_csv('../results/cell_morphology_all.csv')
    print('Saved cell_morphology_all.csv')

## 5. Morphology Comparison Across Cell Types

In [ ]:
if 'df_morph_annotated' in dir():
    # Violin plots: roundness by major cell type
    cell_type_order = ['Tumor', 'Normal', 'Stromal', 'Myeloid', 'T cell', 'B cell']
    df_plot = df_morph_annotated.dropna(subset=['Major_cluster_pathol', 'roundness'])
    df_plot = df_plot[df_plot['Major_cluster_pathol'].isin(cell_type_order)]

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    for ax, metric in zip(axes, ['roundness', 'aspect_ratio', 'area']):
        sns.violinplot(
            data=df_plot,
            x='Major_cluster_pathol', y=metric,
            order=cell_type_order,
            palette='Set2', ax=ax, cut=0
        )
        ax.set_title(metric.replace('_', ' ').title())
        ax.set_xlabel('Cell type')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

    plt.suptitle('Cell Morphology by Major Cell Type', fontsize=13)
    plt.tight_layout()
    plt.savefig('../figures/05_morphology_by_celltype.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Statistical comparison: roundness by response group for tumor cells
if 'df_morph_annotated' in dir():
    df_tumor_morph = df_morph_annotated[
        (df_morph_annotated['Major_cluster_pathol'] == 'Tumor') &
        (df_morph_annotated['Timepoint'] == 'Pre')
    ].dropna(subset=['roundness', 'Response'])

    response_groups = ['SD', 'MPR', 'CR']
    response_colors = {'SD': 'red', 'MPR': 'orange', 'CR': 'blue'}

    # Kruskal-Wallis test
    group_data = [df_tumor_morph.loc[df_tumor_morph['Response'] == r, 'roundness'].dropna() for r in response_groups]
    stat, pval = stats.kruskal(*[g for g in group_data if len(g) > 0])
    print(f'Tumor roundness (Pre): Kruskal-Wallis H={stat:.2f}, p={pval:.4f}')

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.boxplot(
        data=df_tumor_morph, x='Response', y='roundness',
        order=response_groups,
        palette=response_colors, ax=ax
    )
    ax.set_title(f'Tumor Cell Roundness at Pre-treatment\n(Kruskal-Wallis p={pval:.4f})')
    ax.set_ylabel('Roundness (circularity)')
    plt.tight_layout()
    plt.savefig('../figures/05_tumor_roundness_by_response.png', dpi=300, bbox_inches='tight')
    plt.show()

## 6. Morphology by Subcluster

In [ ]:
if 'df_morph_annotated' in dir():
    # CAF subclusters: roundness comparison
    caf_subclusters = [
        'c3_POSTN+_CAF', 'c4_MMP11+_CAF', 'c5_ACTA2+_myCAF',
        'c1_POSTN+_CAF', 'c6_CXCL14+_CAF', 'c0_Epi_CAF_boundary'
    ]
    df_caf_morph = df_morph_annotated[
        df_morph_annotated['subcluster'].isin(caf_subclusters)
    ].dropna(subset=['roundness'])

    if len(df_caf_morph) > 0:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        for ax, metric in zip(axes, ['roundness', 'aspect_ratio', 'area']):
            sns.boxplot(
                data=df_caf_morph, x='subcluster', y=metric,
                order=caf_subclusters, ax=ax
            )
            ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
            ax.set_title(metric.replace('_', ' ').title())

        plt.suptitle('CAF Subcluster Morphology', fontsize=13)
        plt.tight_layout()
        plt.savefig('../figures/05_caf_morphology_subclusters.png', dpi=300, bbox_inches='tight')
        plt.show()

## 7. Longitudinal Morphology Changes

In [ ]:
if 'df_morph_annotated' in dir():
    # Track tumor cell roundness change Pre → Resection per patient
    df_tumor_long = df_morph_annotated[
        df_morph_annotated['Major_cluster_pathol'] == 'Tumor'
    ].groupby(['Patient', 'Timepoint', 'Response'])['roundness'].median().reset_index()

    df_tumor_long['Timepoint'] = pd.Categorical(
        df_tumor_long['Timepoint'],
        categories=['Pre', 'JustAfter', 'Resection'], ordered=True
    )
    df_tumor_long = df_tumor_long.sort_values(['Patient', 'Timepoint'])

    response_colors = {'SD': 'red', 'MPR': 'orange', 'CR': 'blue'}
    fig, ax = plt.subplots(figsize=(7, 5))

    for patient, grp in df_tumor_long.groupby('Patient'):
        response = grp['Response'].iloc[0]
        ax.plot(
            grp['Timepoint'].astype(str), grp['roundness'],
            marker='o', color=response_colors.get(response, 'grey'),
            alpha=0.5, linewidth=1
        )

    ax.set_title('Median Tumor Cell Roundness Over Time')
    ax.set_xlabel('Timepoint')
    ax.set_ylabel('Roundness (median)')

    from matplotlib.lines import Line2D
    legend_elements = [Line2D([0], [0], color=c, label=r) for r, c in response_colors.items()]
    ax.legend(handles=legend_elements, title='Response')

    plt.tight_layout()
    plt.savefig('../figures/05_tumor_roundness_longitudinal.png', dpi=300, bbox_inches='tight')
    plt.show()